# Mobil Uygulama Yorumlarında Müşteri Memnuniyet Analizi

**Veri Madenciliği Dersi — Dönem Projesi**

| Ad Soyad | Öğrenci No |
|----------|------------|
| Furkan Öztürk | 230229083 |
| Taha Yasin Çiçek | 230229088 |
| Ziyaeddin Ayerden | 210229022 |

---

## 1. Giriş

Bu projede, Google Play Store ve Apple App Store'dan toplanan **71.000+** Türkçe müşteri yorumu üzerinde doğal dil işleme (NLP) ve makine öğrenmesi teknikleri kullanılarak otomatik duygu sınıflandırması yapılmaktadır.

### Amaç
Müşteri yorumlarındaki metin verisi kullanılarak memnuniyet düzeyi (pozitif / negatif / nötr) ne düzeyde doğrulukla tahmin edilebileceğini araştırmak.

### Kapsam
- **6 kategori**, **55 uygulama** üzerinden veri toplama
- Metin ön işleme ve özellik çıkarma
- TF-IDF analizi ve kelime bulutları
- Konu modelleme (LDA)
- BERT ile derin öğrenme tabanlı duygu analizi
- Klasik ML modelleri (Logistic Regression, Naive Bayes, SVM, Random Forest, XGBoost) ile karşılaştırma
- LIME ile model yorumlanabilirliği
- Streamlit ile interaktif web arayüzü

### Veri Kaynakları

| Platform | Yöntem | Yorum Sayısı |
|----------|--------|--------------|
| Google Play Store | google-play-scraper | ~49.925 |
| Apple App Store | iTunes RSS API | ~21.075 |

### Kategoriler ve Uygulamalar

| Kategori | Uygulama Sayısı | Örnekler |
|----------|-----------------|----------|
| Sosyal Medya | 10 | YouTube, Instagram, TikTok, WhatsApp |
| E-Ticaret | 10 | Trendyol, Hepsiburada, Amazon, n11 |
| Yemek Siparişi | 8 | Yemeksepeti, Getir, Trendyol Go |
| Market | 9 | BİM, ŞOK, Migros, A101 |
| Kariyer | 6 | LinkedIn, Kariyer.net, İŞKUR |
| Bankacılık | 12 | Ziraat, Akbank, Garanti BBVA |

## 2. Kütüphaneler ve Yapılandırma

In [ ]:
import os
import re
import sys
import string
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from sklearn.decomposition import LatentDirichletAllocation

import nltk
from nltk.corpus import stopwords

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Proje kökü:', PROJECT_ROOT)

## 3. Veri Toplama Metodolojisi

Veri toplama süreci iki farklı platform üzerinden gerçekleştirilmiştir:

**Google Play Store:**
- `google-play-scraper` kütüphanesi kullanıldı
- Her uygulama için 1-5 yıldız aralığında **dengeli örnekleme** yapıldı (her yıldızdan 200 yorum)
- Dil: Türkçe (`lang='tr'`, `country='tr'`)
- Rate limiting ile istekler arası bekleme süresi eklendi

**Apple App Store:**
- iTunes RSS API kullanıldı (kimlik doğrulama gerektirmez)
- JSON feed parse edilerek yorumlar toparlandı
- Sayfalama desteği (max 10 sayfa × 50 yorum)
- Yazar + tarih hash'i ile tekrar eden yorumlar elendi

**Orkestratör (`src/scraping/run_all.py`):**
- 55 uygulamayı sıralıyla işledi
- CSV dosyasına artımlı kayıt yaptı
- `review_id` ve `platform` bazında tekillik kontrolü

Toplam **~71.000** yorum başarıyla toplandı.

## 4. Veri Yükleme ve İlk İnceleme

In [ ]:
app_store = pd.read_csv(RAW_DIR / 'app_store_reviews.csv')
google_play = pd.read_csv(RAW_DIR / 'google_play_reviews.csv')
df = pd.concat([app_store, google_play], ignore_index=True)

print(f'App Store yorum sayısı: {len(app_store):,}')
print(f'Google Play yorum sayısı: {len(google_play):,}')
print(f'Toplam yorum sayısı: {len(df):,}')
print()
df.info()

In [ ]:
df.sample(5)

In [ ]:
df.describe()

In [ ]:
print('Eksik değer sayıları:')
print(df.isnull().sum())
print()
print('Tekrar eden satır sayısı:', df.duplicated().sum())

## 5. Veri Temizleme ve Ön İşleme

In [ ]:
df['text'] = df['text'].fillna('').astype(str)
df['title'] = df['title'].fillna('').astype(str)
df['full_text'] = (df['title'] + ' ' + df['text']).str.strip()
df = df[df['full_text'].str.len() > 0].copy()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(int)

print(f'Temizleme sonrası toplam yorum: {len(df):,}')
print(f'Platform dağılımı:')
print(df['platform'].value_counts())

### 5.1 Duygu Etiketleme

Yıldız puanlarına göre 3 sınıflı duygu etiketi atanır:
- **1-2 yıldız** → Negatif
- **3 yıldız** → Nötr
- **4-5 yıldız** → Pozitif

In [ ]:
def label_sentiment(rating: int) -> str:
    if rating <= 2:
        return 'negatif'
    if rating == 3:
        return 'nötr'
    return 'pozitif'

df['sentiment'] = df['rating'].apply(label_sentiment)
print('Duygu dağılımı:')
print(df['sentiment'].value_counts())

### 5.2 Metin Temizleme

Küçük harfe çevirme, URL/sayı/noktalama kaldırma, stopword'leri ve 2 karakterden kısa token'ları silme.

In [ ]:
try:
    TR_STOPWORDS = set(stopwords.words('turkish'))
except LookupError:
    nltk.download('stopwords')
    TR_STOPWORDS = set(stopwords.words('turkish'))

EXTRA_STOPWORDS = {
    'bir', 'çok', 'daha', 'şey', 'şu', 'şöyle', 'şimdi', 'bende', 'bana',
    'sonra', 'önce', 'kadar', 'gibi', 'oluyor', 'oldu', 'olmuş',
    'var', 'yok', 'evet', 'hayır', 'ama', 'fakat', 'tam', 'hep', 'hiç',
    'uygulama', 'uygulamayı', 'uygulamanın', 'uygulamada',
    'telefon', 'telefonum', 'telefonumda',
    'app', 'video', 'videolar', 'youtube', 'instagram', 'whatsapp', 'tiktok',
    'için', 'ile', 'ki', 'mi', 'mı', 'mu', 'mü', 'da', 'de', 'ta', 'te'
}
STOPWORDS = TR_STOPWORDS | EXTRA_STOPWORDS

URL_RE = re.compile(r'https?://\S+|www\.\S+')
NON_WORD_RE = re.compile(r'[^a-zçğıöşü\s]+', flags=re.IGNORECASE)
MULTI_SPACE_RE = re.compile(r'\s+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = text.lower()
    t = t.replace('İ', 'i').replace('I', 'ı')
    t = URL_RE.sub(' ', t)
    t = NON_WORD_RE.sub(' ', t)
    t = MULTI_SPACE_RE.sub(' ', t).strip()
    tokens = [w for w in t.split() if len(w) > 2 and w not in STOPWORDS]
    return ' '.join(tokens)

df['clean'] = df['full_text'].apply(clean_text)
df = df[df['clean'].str.len() > 0].copy()

print('Temizleme sonrası örnek:')
df[['rating', 'sentiment', 'full_text', 'clean']].head(5)

### 5.3 Özellik Mühendisliği (Feature Engineering)

In [ ]:
df['char_count'] = df['clean'].apply(len)
df['word_count'] = df['clean'].apply(lambda x: len(x.split()))

print('Karakter ve kelime sayısı istatistikleri:')
df[['char_count', 'word_count']].describe()

---

## 6. Keşifsel Veri Analizi (EDA)

Bu bölümde verinin genel yapısını, dağılımlarını ve örüntülerini görsel olarak inceliyoruz.

### 6.1 Puan ve Duygu Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order_r = sorted(df['rating'].unique())
sns.countplot(data=df, x='rating', order=order_r, palette='viridis', ax=axes[0])
axes[0].set_title('Yıldız Puanı Dağılımı')
axes[0].set_xlabel('Yıldız')
axes[0].set_ylabel('Yorum Sayısı')

order_s = ['negatif', 'nötr', 'pozitif']
colors_s = ['#d62728', '#7f7f7f', '#2ca02c']
sns.countplot(data=df, x='sentiment', order=order_s, palette=colors_s, ax=axes[1])
axes[1].set_title('Duygu Sınıfı Dağılımı')
axes[1].set_xlabel('Sınıf')
axes[1].set_ylabel('Yorum Sayısı')

plt.tight_layout()
plt.savefig(VISUALS_DIR / 'sentiment_distribution.png', bbox_inches='tight')
plt.show()

### 6.2 Duygu Dağılımı — Pasta Grafiği

In [ ]:
label_data = df['sentiment'].value_counts()
explode = (0.05, 0.05, 0.05)
colors_pie = ['#d62728', '#7f7f7f', '#2ca02c']

fig, ax = plt.subplots(figsize=(8, 6))
patches, texts, pcts = ax.pie(
    label_data, labels=label_data.index, explode=explode,
    autopct='%1.1f%%', shadow=True, startangle=90,
    colors=colors_pie,
    textprops={'fontsize': 11, 'weight': 'bold'}
)
plt.setp(pcts, color='white')
centre = plt.Circle((0, 0), 0.40, fc='white')
fig.gca().add_artist(centre)
ax.set_title('Duygu Sınıfı Oranı', fontsize=14)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'sentiment_pie.png', bbox_inches='tight')
plt.show()

### 6.3 Platform Bazında Duygu Dağılımı

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
hue_order = ['pozitif', 'nötr', 'negatif']
sns.countplot(data=df, x='platform', hue='sentiment', hue_order=hue_order,
              palette=['#2ca02c', '#7f7f7f', '#d62728'], ax=ax)
ax.set_title('Platform Bazında Duygu Dağılımı')
ax.set_xlabel('Platform')
ax.set_ylabel('Yorum Sayısı')
ax.legend(title='Duygu')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'platform_sentiment.png', bbox_inches='tight')
plt.show()

### 6.4 En Çok Yorum Alan Uygulamalar

In [ ]:
top_apps = df['app_name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
top_apps.plot(kind='barh', color=sns.color_palette('viridis', len(top_apps)), ax=ax)
ax.set_title('En Çok Yorum Alan 15 Uygulama')
ax.set_xlabel('Yorum Sayısı')
ax.set_ylabel('Uygulama')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'top_apps.png', bbox_inches='tight')
plt.show()

### 6.5 Uygulama Bazında Ortalama Puan

In [ ]:
app_ratings = df.groupby('app_name')['rating'].mean().sort_values()

fig, ax = plt.subplots(figsize=(14, 8))
app_ratings.plot(kind='barh', color=plt.cm.RdYlGn(np.linspace(0, 1, len(app_ratings))), ax=ax)
ax.set_title('Uygulama Bazında Ortalama Yıldız Puanı')
ax.set_xlabel('Ortalama Puan')
ax.set_ylabel('Uygulama')
ax.axvline(x=3, color='gray', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'app_avg_rating.png', bbox_inches='tight')
plt.show()

### 6.6 Yorum Uzunluğu Analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['word_count'], kde=True, bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Kelime Sayısı Dağılımı')
axes[0].set_xlabel('Kelime Sayısı')
axes[0].set_ylabel('Frekans')

sns.boxplot(data=df, x='sentiment', y='word_count',
            order=['negatif', 'nötr', 'pozitif'],
            palette=['#d62728', '#7f7f7f', '#2ca02c'], ax=axes[1])
axes[1].set_title('Duygu Sınıfına Göre Kelime Sayısı')
axes[1].set_xlabel('Duygu')
axes[1].set_ylabel('Kelime Sayısı')

plt.tight_layout()
plt.savefig(VISUALS_DIR / 'word_count_analysis.png', bbox_inches='tight')
plt.show()

### 6.7 Karakter Sayısı vs Kelime Sayısı

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(x='char_count', y='word_count', data=df, alpha=0.3, ax=ax)
ax.set_xlabel('Karakter Sayısı')
ax.set_ylabel('Kelime Sayısı')
ax.set_title('Karakter Sayısı vs Kelime Sayısı')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'char_vs_word.png', bbox_inches='tight')
plt.show()

### 6.8 Kelime Frekansı Analizi

In [ ]:
all_words = ' '.join(df['clean']).split()
word_freq = Counter(all_words).most_common(30)

fig, ax = plt.subplots(figsize=(14, 7))
words = [w[0] for w in word_freq]
counts = [w[1] for w in word_freq]
sns.barplot(x=counts, y=words, palette='viridis', ax=ax)
ax.set_title('En Sık Kullanılan 30 Kelime')
ax.set_xlabel('Frekans')
ax.set_ylabel('Kelime')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'top_30_words.png', bbox_inches='tight')
plt.show()

### 6.9 Korelasyon Analizi

In [ ]:
numeric_df = df[['rating', 'char_count', 'word_count']]
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_title('Sayısal Değişkenler Arası Korelasyon')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'correlation_matrix.png', bbox_inches='tight')
plt.show()

---

## 7. TF-IDF Analizi ve Kelime Bulutları

Her yorum bir doküman olarak ele alınır. `TfidfVectorizer` tüm korpus üzerinde fit edilir, ardından her duygu sınıfı için ortalama TF-IDF skoru hesaplanır. Yüksek ortalama skor, o sınıfa özgü kelimeleri gösterir.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.85,
)
X_tfidf = vectorizer.fit_transform(df['clean'])
vocab = np.array(vectorizer.get_feature_names_out())
print('Sözlük boyutu:', len(vocab))
print('Matris şekli:', X_tfidf.shape)

In [ ]:
def class_mean_tfidf(sentiment_label: str) -> pd.Series:
    mask = (df['sentiment'] == sentiment_label).values
    if mask.sum() == 0:
        return pd.Series(dtype=float)
    means = np.asarray(X_tfidf[mask].mean(axis=0)).ravel()
    return pd.Series(means, index=vocab).sort_values(ascending=False)

TOP_N = 30
top_pos = class_mean_tfidf('pozitif').head(TOP_N)
top_neg = class_mean_tfidf('negatif').head(TOP_N)
top_neu = class_mean_tfidf('nötr').head(TOP_N)

top_pos.to_csv(PROCESSED_DIR / 'top_words_positive.csv', header=['tfidf'])
top_neg.to_csv(PROCESSED_DIR / 'top_words_negative.csv', header=['tfidf'])
top_neu.to_csv(PROCESSED_DIR / 'top_words_neutral.csv', header=['tfidf'])

comparison = pd.DataFrame({
    'pozitif_kelime': top_pos.head(20).index,
    'pozitif_tfidf': top_pos.head(20).values,
    'negatif_kelime': top_neg.head(20).index,
    'negatif_tfidf': top_neg.head(20).values,
})
comparison

### 7.1 Pozitif vs Negatif Öne Çıkan Kelimeler

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

top_pos.head(20).iloc[::-1].plot(kind='barh', color='#2ca02c', ax=axes[0])
axes[0].set_title('Pozitif Yorumlarda Öne Çıkan Kelimeler (TF-IDF)')
axes[0].set_xlabel('Ortalama TF-IDF')

top_neg.head(20).iloc[::-1].plot(kind='barh', color='#d62728', ax=axes[1])
axes[1].set_title('Negatif Yorumlarda Öne Çıkan Kelimeler (TF-IDF)')
axes[1].set_xlabel('Ortalama TF-IDF')

plt.tight_layout()
plt.savefig(VISUALS_DIR / 'top_words_pos_neg.png', bbox_inches='tight')
plt.show()

### 7.2 Kelime Bulutları

In [ ]:
def make_wordcloud(freqs, colormap, title, filename):
    wc = WordCloud(
        width=1400, height=700,
        background_color='white',
        colormap=colormap,
        prefer_horizontal=0.9,
        random_state=42,
    ).generate_from_frequencies(freqs.to_dict())
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / filename, bbox_inches='tight', dpi=150)
    plt.show()

pos_freqs = class_mean_tfidf('pozitif').head(150)
neg_freqs = class_mean_tfidf('negatif').head(150)

make_wordcloud(pos_freqs, 'Greens', 'Pozitif Yorumlarda Öne Çıkan Kelimeler', 'wordcloud_positive.png')
make_wordcloud(neg_freqs, 'Reds', 'Negatif Yorumlarda Öne Çıkan Kelimeler', 'wordcloud_negative.png')

### 7.3 Genel Kelime Bulutu

In [ ]:
all_text = ' '.join(df['clean'])
wc_all = WordCloud(width=1200, height=600, background_color='white',
                   colormap='plasma', random_state=42).generate(all_text)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc_all, interpolation='bilinear')
ax.axis('off')
ax.set_title('Tüm Yorumlarda Kelime Bulutu', fontsize=14)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'wordcloud_all.png', bbox_inches='tight', dpi=150)
plt.show()

---

## 8. Konu Modelleme (LDA)

Latent Dirichlet Allocation (LDA) ile müşteri yorumlarındaki gizli konuları/temaları ortaya çıkarıyoruz.

In [ ]:
cv_lda = CountVectorizer(max_features=2000, ngram_range=(1, 2))
dtm = cv_lda.fit_transform(df['clean'])

num_topics = 10
lda = LatentDirichletAllocation(n_components=num_topics, random_state=42, n_jobs=-1)
lda.fit(dtm)

feature_names = cv_lda.get_feature_names_out()
top_words = 10

print('=== LDA Konu Modelleme Sonuçları ===')
print()
for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[:-top_words - 1:-1]
    top_terms = [feature_names[i] for i in top_indices]
    print(f'Konu {topic_idx + 1}: {" | ".join(top_terms)}')

---

## 9. BERT ile Duygu Analizi

**Model:** `nlptown/bert-base-multilingual-uncased-sentiment`
- Milyonlarca yorumla ince ayarlı, 1-5 yıldız tahmini yapar
- Türkçe dahil 6 dilde destek
- 5-sınıflı yıldız tahminleri 3-sınıflı duygu etiketlerine dönüştürülür

CPU üzerinde tüm verinin işlenmesi çok uzun süreceği için, her yıldız sınıfından eşit sayıda örneklem alınarak dengeli bir test seti oluşturulur.

In [ ]:
from src.models.bert import (
    SENTIMENT_CLASSES, STAR_TO_SENTIMENT,
    load_model, predict_stars, stars_to_3class_probs, stars_to_sentiment,
)
from tqdm.auto import tqdm

tokenizer, model_bert, device = load_model()
print('Cihaz:', device)
print('Model:', model_bert.config.name_or_path)

In [ ]:
PER_CLASS = 600
RANDOM_STATE = 42

df['true_sentiment'] = df['rating'].map(STAR_TO_SENTIMENT)

sample = (
    df.groupby('rating', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), PER_CLASS), random_state=RANDOM_STATE))
      .reset_index(drop=True)
)
print(f'Örneklem boyutu: {len(sample):,}')
sample['rating'].value_counts().sort_index()

In [ ]:
texts = sample['full_text'].tolist()
BATCH = 16

all_probs = []
for i in tqdm(range(0, len(texts), BATCH), desc='BERT tahmin'):
    batch = texts[i:i + BATCH]
    p, _ = predict_stars(batch, batch_size=BATCH)
    all_probs.append(p)

star_probs = np.vstack(all_probs)
pred_stars = star_probs.argmax(axis=1) + 1
sentiment_probs = stars_to_3class_probs(star_probs)
pred_sentiments = stars_to_sentiment(pred_stars)

sample = sample.assign(
    pred_star=pred_stars,
    pred_sentiment=pred_sentiments,
    prob_negatif=sentiment_probs[:, 0],
    prob_notr=sentiment_probs[:, 1],
    prob_pozitif=sentiment_probs[:, 2],
)
sample[['app_name', 'rating', 'pred_star', 'true_sentiment', 'pred_sentiment']].head()

### 9.1 BERT Değerlendirme — 5 Sınıf Yıldız

In [ ]:
y_true = sample['rating'].values
y_pred = sample['pred_star'].values

acc = accuracy_score(y_true, y_pred)
print(f'5-sınıf accuracy: {acc:.3f}')
print()
print(classification_report(y_true, y_pred, digits=3))

cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[1, 2, 3, 4, 5], yticklabels=[1, 2, 3, 4, 5], ax=ax)
ax.set_xlabel('Tahmin (yıldız)')
ax.set_ylabel('Gerçek (yıldız)')
ax.set_title(f'BERT 5-Sınıf Confusion Matrix (Accuracy: {acc:.3f})')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'bert_confusion_5class.png', bbox_inches='tight')
plt.show()

### 9.2 BERT Değerlendirme — 3 Sınıf Duygu

In [ ]:
y_true_s = sample['true_sentiment'].values
y_pred_s = sample['pred_sentiment'].values

acc3 = accuracy_score(y_true_s, y_pred_s)
print(f'3-sınıf accuracy: {acc3:.3f}')
print()
print(classification_report(y_true_s, y_pred_s, labels=SENTIMENT_CLASSES, digits=3))

cm3 = confusion_matrix(y_true_s, y_pred_s, labels=SENTIMENT_CLASSES)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm3, annot=True, fmt='d', cmap='Greens',
            xticklabels=SENTIMENT_CLASSES, yticklabels=SENTIMENT_CLASSES, ax=ax)
ax.set_xlabel('Tahmin')
ax.set_ylabel('Gerçek')
ax.set_title(f'BERT 3-Sınıf Sentiment Confusion (Accuracy: {acc3:.3f})')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'bert_confusion_3class.png', bbox_inches='tight')
plt.show()

### 9.3 Doğru ve Yanlış Tahmin Örnekleri

In [ ]:
display_cols = ['app_name', 'rating', 'pred_star', 'true_sentiment', 'pred_sentiment', 'full_text']

print('Doğru tahmin örnekleri:')
correct = sample[sample['true_sentiment'] == sample['pred_sentiment']].sample(5, random_state=1)
display(correct[display_cols])

print('Yanlış tahmin örnekleri:')
wrong = sample[sample['true_sentiment'] != sample['pred_sentiment']]
if len(wrong) > 0:
    display(wrong.sample(min(5, len(wrong)), random_state=1)[display_cols])

In [ ]:
out_path = PROCESSED_DIR / 'bert_predictions.csv'
save_cols = [
    'review_id', 'platform', 'app_name', 'rating',
    'true_sentiment', 'pred_star', 'pred_sentiment',
    'prob_negatif', 'prob_notr', 'prob_pozitif', 'full_text',
]
sample[save_cols].to_csv(out_path, index=False)
print(f'BERT tahminleri kaydedildi: {out_path} ({len(sample):,} satır)')

---

## 10. Klasik Makine Öğrenmesi Modelleri

BERT'e ek olarak, geleneksel ML modellerini de eğitip karşılaştırıyoruz. Modeller TF-IDF özellik vektörleri üzerinde çalışır.

**Test edilen modeller:**
1. Logistic Regression
2. Multinomial Naive Bayes
3. Support Vector Machine (SVM)
4. Random Forest
5. XGBoost

In [ ]:
le = LabelEncoder()
X_ml = df['clean']
y_ml = df['sentiment']
y_encoded = le.fit_transform(y_ml)

label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Etiket eşlemesi:', label_mapping)
print(f'Toplam veri: {len(X_ml):,}')

X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_encoded, stratify=y_encoded, test_size=0.2, random_state=42
)
print(f'Eğitim seti: {len(X_train):,}')
print(f'Test seti: {len(X_test):,}')

In [ ]:
preprocessor = Pipeline([
    ('bow', CountVectorizer(ngram_range=(1, 2), max_features=10000)),
    ('tfidf', TfidfTransformer()),
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(),
    'SVM': SVC(kernel='linear', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=7, eta=0.1,
        objective='multi:softmax', num_class=3,
        eval_metric='mlogloss', random_state=42,
        use_label_encoder=False,
    ),
}

results = []
predictions = {}

for name, model in models.items():
    print(f'\n{"=" * 50}')
    print(f'{name} eğitiliyor...')
    print(f'{"=" * 50}')
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    predictions[name] = y_pred
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
    })
    
    print(f'Accuracy: {acc:.4f}')
    print(f'Precision: {prec:.4f}')
    print(f'Recall: {rec:.4f}')
    print(f'F1-Score: {f1:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
print('\n=== MODEL KARŞILAŞTIRMA TABLOSU ===')
display(results_df)

### 10.1 Confusion Matrix'ler

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[idx])
    acc = accuracy_score(y_test, y_pred)
    axes[idx].set_title(f'{name}\nAccuracy: {acc:.3f}')
    axes[idx].set_xlabel('Tahmin')
    axes[idx].set_ylabel('Gerçek')

axes[-1].axis('off')
plt.suptitle('Klasik ML Modelleri — Confusion Matrix Karşılaştırması', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'ml_confusion_matrices.png', bbox_inches='tight')
plt.show()

### 10.2 Model Performans Karşılaştırması

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(results_df))
width = 0.2

ax.bar(x_pos - 1.5*width, results_df['Accuracy'], width, label='Accuracy', color='#1f77b4')
ax.bar(x_pos - 0.5*width, results_df['Precision'], width, label='Precision', color='#ff7f0e')
ax.bar(x_pos + 0.5*width, results_df['Recall'], width, label='Recall', color='#2ca02c')
ax.bar(x_pos + 1.5*width, results_df['F1-Score'], width, label='F1-Score', color='#d62728')

ax.set_xlabel('Model')
ax.set_ylabel('Skor')
ax.set_title('Klasik ML Modelleri Performans Karşılaştırması')
ax.set_xticks(x_pos)
ax.set_xticklabels(results_df['Model'], rotation=15)
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(VISUALS_DIR / 'ml_model_comparison.png', bbox_inches='tight')
plt.show()

### 10.3 Çapraz Doğrulama (Cross-Validation)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    scores = cross_val_score(pipeline, X_ml, y_encoded, cv=kf, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'CV Ortalama Accuracy': scores.mean(),
        'CV Std': scores.std(),
    })
    print(f'{name}: {scores.mean():.4f} (+/- {scores.std():.4f})')

cv_df = pd.DataFrame(cv_results).sort_values('CV Ortalama Accuracy', ascending=False)
print('\n=== ÇAPRAZ DOĞRULAMA SONUÇLARI ===')
display(cv_df)

---

## 11. Tüm Modellerin Genel Karşılaştırması

BERT ve klasik ML modellerinin performanslarını bir arada değerlendirelim.

In [ ]:
all_results = results_df.copy()
bert_row = pd.DataFrame([{
    'Model': 'BERT (multilingual)',
    'Accuracy': acc3,
    'Precision': float(classification_report(
        y_true_s, y_pred_s, labels=SENTIMENT_CLASSES, output_dict=True)['weighted avg']['precision']),
    'Recall': float(classification_report(
        y_true_s, y_pred_s, labels=SENTIMENT_CLASSES, output_dict=True)['weighted avg']['recall']),
    'F1-Score': float(classification_report(
        y_true_s, y_pred_s, labels=SENTIMENT_CLASSES, output_dict=True)['weighted avg']['f1-score']),
}])
all_results = pd.concat([all_results, bert_row], ignore_index=True).sort_values('F1-Score', ascending=False)

print('=== TÜM MODELLER — PERFORMANS KARŞILAŞTIRMASI ===')
display(all_results)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#ff6b6b' if 'BERT' in m else '#4ecdc4' for m in all_results['Model']]
all_results.plot(x='Model', y='F1-Score', kind='bar', color=colors, ax=ax, legend=False)
ax.set_title('Tüm Modeller — F1-Score Karşılaştırması', fontsize=14)
ax.set_xlabel('Model')
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.05)
for i, (_, row) in enumerate(all_results.iterrows()):
    ax.text(i, row['F1-Score'] + 0.01, f'{row["F1-Score"]:.3f}', ha='center', fontsize=10)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'all_models_comparison.png', bbox_inches='tight')
plt.show()

---

## 12. LIME ile Model Yorumlanabilirliği

BERT bir yorumu sınıflandırırken **hangi kelimelerin** kararı etkilediğini LIME (Local Interpretable Model-agnostic Explanations) ile gösteriyoruz.

- **Yeşil** = pozitif yönde katkı
- **Kırmızı** = negatif yönde katkı

LIME, modeli yerel olarak doğrusal bir vekil ile yaklaşık tahmin eder: bir yorumdaki kelimeleri tek tek silip modelin olasılık çıktısının nasıl değiştiğine bakar.

In [ ]:
from lime.lime_text import LimeTextExplainer
from src.models.bert import predict_3class
from IPython.display import HTML, display

EXPL_DIR = VISUALS_DIR / 'lime'
EXPL_DIR.mkdir(parents=True, exist_ok=True)

explainer = LimeTextExplainer(class_names=SENTIMENT_CLASSES, bow=False)

def predict_fn(texts):
    return predict_3class(list(texts), batch_size=16)

print('LIME hazırlandı.')

In [ ]:
df_lime = df[df['full_text'].str.split().str.len().between(8, 40)].copy()

lime_samples = pd.concat([
    df_lime[df_lime['rating'] <= 2].sample(2, random_state=7),
    df_lime[df_lime['rating'] == 3].sample(2, random_state=7),
    df_lime[df_lime['rating'] >= 4].sample(2, random_state=7),
]).reset_index(drop=True)

print('Seçilen örnekler:')
lime_samples[['app_name', 'rating', 'full_text']]

In [ ]:
NUM_FEATURES = 10
NUM_SAMPLES = 200

for i, row in lime_samples.iterrows():
    text = row['full_text']
    print(f'--- Örnek {i+1} | Gerçek yıldız: {row["rating"]} | Uygulama: {row["app_name"]} ---')
    print(text)
    probs = predict_fn([text])[0]
    pred_idx = int(np.argmax(probs))
    print(f'Tahmin: {SENTIMENT_CLASSES[pred_idx]} '
          f'(neg={probs[0]:.2f}, nötr={probs[1]:.2f}, poz={probs[2]:.2f})')

    exp = explainer.explain_instance(
        text, predict_fn,
        num_features=NUM_FEATURES,
        num_samples=NUM_SAMPLES,
        labels=[0, 2],
    )
    print('  En etkili kelimeler (pozitif sınıf için):')
    for word, weight in exp.as_list(label=2):
        print(f'    {weight:+.3f}  {word}')

    html_path = EXPL_DIR / f'example_{i+1}_rating{row["rating"]}.html'
    exp.save_to_file(str(html_path))
    display(HTML(exp.as_html(labels=[2])))
    print()

### 12.1 Toplu Kelime Ağırlıkları

In [ ]:
AGG_PER_CLASS = 10

agg_samples = pd.concat([
    df_lime[df_lime['rating'] <= 2].sample(AGG_PER_CLASS, random_state=1),
    df_lime[df_lime['rating'] >= 4].sample(AGG_PER_CLASS, random_state=1),
])

word_weights = defaultdict(list)
for text in agg_samples['full_text']:
    exp = explainer.explain_instance(
        text, predict_fn,
        num_features=15, num_samples=150, labels=[2],
    )
    for word, w in exp.as_list(label=2):
        word_weights[word.lower()].append(w)

agg = pd.DataFrame([
    {'kelime': w, 'ortalama_ağırlık': np.mean(ws), 'gözlem': len(ws)}
    for w, ws in word_weights.items() if len(ws) >= 2
]).sort_values('ortalama_ağırlık')

print('En negatif sinyaller (model tarafından):')
display(agg.head(15))
print('\nEn pozitif sinyaller (model tarafından):')
display(agg.tail(15))

agg.to_csv(PROCESSED_DIR / 'lime_aggregate_weights.csv', index=False)

---

## 13. Yeni Örnekler Üzerinde Tahmin

Eğitilen modellerin gerçek dünyadan örnekler üzerindeki performansını test edelim.

In [ ]:
from src.models.bert import predict_sentiment

test_reviews = [
    'Bu uygulama harika, çok beğendim. Her şey çok kolay ve hızlı.',
    'Rezalet bir uygulama, sürekli çöküyor ve paramızı çalıyorlar.',
    'Fena değil ama bazı özellikler eksik, geliştirilmeli.',
    'Müşteri hizmeti berbat, kimse yardımcı olmuyor.',
    'Güzel tasarım, kullanıcı dostu arayüz. Tavsiye ederim.',
]

print('=== BERT ile Yeni Tahminler ===')
for review in test_reviews:
    result = predict_sentiment(review)
    print(f'\nYorum: "{review}"')
    print(f'  Tahmin: {result["sentiment"]} ({result["sentiment_confidence"]:.2%})')
    print(f'  Yıldız: {result["star"]}')

---

## 14. Sonuçlar ve Değerlendirme

### Temel Bulgular

1. **Veri Seti:** Google Play ve App Store'dan 6 kategoride 55 uygulamadan toplam ~71.000 Türkçe yorum başarıyla toplanmıştır.

2. **TF-IDF Analizi:** Pozitif ve negatif yorumlarda belirgin kelime farklılıkları tespit edilmiştir. Negatif yorumlarda şikayete yönelik, pozitif yorumlarda memnuniyet ifadeleri öne çıkmaktadır.

3. **Konu Modelleme (LDA):** 10 farklı tema belirlenmiş olup, müşterilerin en çok uygulamanın performansı, müşteri hizmeti ve fiyatlandırma konularında yorum yaptığı görülmüştür.

4. **Model Karşılaştırması:**
   - BERT (multilingual), derin öğrenme tabanlı yaklaşımı ile en yüksek doğruluk oranına ulaşmıştır
   - Klasik ML modelleri arasında Logistic Regression ve SVM başarılı sonuçlar vermiştir
   - XGBoost ve Random Forest ağaç tabanlı modeller de rekabetçi performans göstermiştir

5. **LIME Yorumlanabilirliği:** Modelin karar mekanizması şeffaf hale getirilmiş, hangi kelimelerin duygu sınıflandırmasında belirleyici olduğu ortaya konmuştur.

### Kısıtlamalar

- BERT modeli önceden eğitilmiş olup, Türkçe'ye özgü ince ayar yapılmamıştır
- Veri seti belirli kategorilerdeki uygulamalarla sınırlıdır
- Yıldız puanlarına dayalı etiketleme, gerçek duyguyu tam yansıtmayabilir
- CPU üzerinde BERT tahminleri yavaş çalışır, GPU ile hızlandırma önerilir

### Gelecek Çalışma Önerileri

- Türkçe'ye özgü BERT modeli (örn. BERTurk) ile ince ayar yapılması
- Daha geniş bir uygulama ve kategori yelpazesi ile veri toplama
- Zaman serisi analizi ile duygu trendlerinin takibi
- Daha gelişmiş ön işleme (lemmatization, stemming) teknikleri
- Aspect-Based Sentiment Analysis ile alt-konu bazlı analiz

---

## 15. İnteraktif Web Arayüzü (Streamlit)

Projenin interaktif olarak kullanılabilmesi için Streamlit tabanlı bir web arayüzü geliştirilmiştir.

**Başlatma komutu:**
```bash
streamlit run src/app/streamlit_app.py
```

**Özellikler:**
- Serbest metin girişi ile anlık duygu tahmini
- BERT modeli ile 5-sınıf yıldız ve 3-sınıf duygu olasılık dağılımı
- Uygulama arama ve yorum filtreleme
- Özet istatistikler ve en iyi/en kötü yorumlar

---

*Bu notebook, Veri Madenciliği Dersi dönem projesi kapsamında hazırlanmıştır.*

*Furkan Öztürk | Taha Yasin Çiçek | Ziyaeddin Ayerden*